DEEP LEARNING BASED SEGMENTATION OF EDENTULOUS REGION IN DENTAL CBCTS

INSTALLATIONS

In [11]:
!pip install monai[all] nibabel SimpleITK pydicom pynrrd tqdm


IMPORTS

In [12]:
!pip install ipywidgets
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import nrrd
from tqdm import tqdm
import torch

from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityRanged, CropForegroundd, RandCropByLabelClassesd,
    RandFlipd, RandRotate90d, RandScaleIntensityd, RandShiftIntensityd,
    EnsureTyped, Compose
)
from monai.data import CacheDataset, DataLoader
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference


DATA PATHS

In [13]:
CBCT_DICOM_DIR = "/workspace/FINAL_CBCTS/"
SEGMENTED_NRRD_DIR = "/workspace/FINAL_SEGMENTED"
ANNOTATED_NRRD_DIR = "/workspace/FINAL_ANNOTATED"

VOLUME_NIFTI_DIR = "/workspace/VOLUME"
SEGMENTED_NIFTI_DIR = "/workspace/SEGMENTED"
ANNOTATED_NIFTI_DIR = "/workspace/ANNOTATED"
MERGED_LABELS_DIR = "/workspace/MERGEDLABELS"

for d in [VOLUME_NIFTI_DIR, SEGMENTED_NIFTI_DIR, ANNOTATED_NIFTI_DIR, MERGED_LABELS_DIR]:
    os.makedirs(d, exist_ok=True)

CHECK FOR DICOMS [SUBFOLDERS CASE]

In [14]:
def find_dicom_series_folder(patient_folder):
    # DIRECTLY DICOMS AVAILABLE
    if glob.glob(os.path.join(patient_folder, "*.dcm")):
        return patient_folder
    
    # LOOK FOR SUB FOLDER
    for sub in os.listdir(patient_folder):
        sub_path = os.path.join(patient_folder, sub)
        if os.path.isdir(sub_path) and glob.glob(os.path.join(sub_path, "*.dcm")):
            return sub_path

    # NOT FOUND
    return None


DICOM TO NIFTI

In [15]:
def convert_dicom_folder_to_nifti(dicom_folder, output_file):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(dicom_folder)
    reader.SetFileNames(dicom_names)
    image = reader.Execute()
    sitk.WriteImage(image, output_file)
    print(f"Saved NIfTI: {output_file}")

for patient in os.listdir(CBCT_DICOM_DIR):
    patient_folder = os.path.join(CBCT_DICOM_DIR, patient)
    if not os.path.isdir(patient_folder):
        print(f"Skipping {patient}: Not a folder")
        continue

    dicom_series_folder = find_dicom_series_folder(patient_folder)
    if dicom_series_folder is None:
        print(f"Skipping {patient}: No DICOM files found in folder or subfolders")
        continue

    out_file = os.path.join(VOLUME_NIFTI_DIR, f"{patient}_volume.nii.gz")
    if not os.path.exists(out_file):
        try:
            convert_dicom_folder_to_nifti(dicom_series_folder, out_file)
        except Exception as e:
            print(f"Error converting {patient}: {e}")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/workspace/FINAL_CBCTS/'

.nrrd to .nii.gz

In [ ]:
for folder in os.listdir(SEGMENTED_NRRD_DIR):
    folder_path = os.path.join(SEGMENTED_NRRD_DIR, folder)
    if os.path.isdir(folder_path):
        print(f"Entering folder: {folder_path}")
        for file in os.listdir(folder_path):
            if file.endswith(".nrrd"):
                in_path = os.path.join(folder_path, file)
                out_name = folder + ".nii.gz"
                out_path = os.path.join(SEGMENTED_NIFTI_DIR, out_name)

                print(f"Converting {in_path} to {out_path}")

                if not os.path.exists(out_path):
                    convert_nrrd_to_nifti(in_path, out_path)
                else:
                    print(f"Skipping, already exists: {out_path}")

for folder in os.listdir(ANNOTATED_NRRD_DIR):
    folder_path = os.path.join(ANNOTATED_NRRD_DIR, folder)
    if os.path.isdir(folder_path):
        print(f"Entering folder: {folder_path}")
        for file in os.listdir(folder_path):
            if file.endswith(".nrrd"):
                in_path = os.path.join(folder_path, file)
                out_name = folder + "_edentulous.nii.gz"
                out_path = os.path.join(ANNOTATED_NIFTI_DIR, out_name)

                print(f"Converting {in_path} to {out_path}")

                if not os.path.exists(out_path):
                    convert_nrrd_to_nifti(in_path, out_path)
                else:
                    print(f"Skipping, already exists: {out_path}")


MERGE SEGMENTED + ANNOTATED LABELS

In [ ]:
import nibabel as nib
import numpy as np
import os

def merge_dental_and_edentulous(dental_seg_path, edentulous_mask_path, output_label_path):
    # Load both segmentations
    dental_seg = nib.load(dental_seg_path)
    edentulous_seg = nib.load(edentulous_mask_path)

    dental_data = dental_seg.get_fdata().astype(np.uint8)
    edentulous_data = edentulous_seg.get_fdata().astype(np.uint8)

    # Ensure matching shapes
    if dental_data.shape != edentulous_data.shape:
        raise ValueError(f"Shape mismatch! Dental: {dental_data.shape}, Edentulous: {edentulous_data.shape}")

    # Merge: assign label 6 to edentulous regions
    merged = np.copy(dental_data)
    merged[edentulous_data > 0] = 6

    # Save new NIfTI file
    nib.save(nib.Nifti1Image(merged, affine=dental_seg.affine), output_label_path)
    print(f"Saved merged label map: {output_label_path}")
          
for seg_file in sorted(os.listdir(SEGMENTED_NIFTI_DIR)):
    if not seg_file.endswith(".nii.gz"):
        continue

    pid = seg_file.replace('.nii.gz', '')
    dental_path = os.path.join(SEGMENTED_NIFTI_DIR, seg_file)
    edentulous_path = os.path.join(ANNOTATED_NIFTI_DIR, f"{pid}_edentulous.nii.gz")
    output_path = os.path.join(MERGED_LABELS_DIR, f"{pid}_label_map.nii.gz")

    if not os.path.exists(dental_path):
        print(f"Skipping {pid}: Dental segmentation file is missing.")
        continue

    if not os.path.exists(edentulous_path):
        print(f"Skipping {pid}: Edentulous mask file is missing.")
        continue

    merge_dental_and_edentulous(dental_path, edentulous_path, output_path)


MONAI DATASET

In [ ]:
data_list = []
for label_file in os.listdir(MERGED_LABELS_DIR):
    pid = label_file.split('_')[0]
    image_file = os.path.join(VOLUME_NIFTI_DIR, f"{pid}_volume.nii.gz")
    label_file = os.path.join(MERGED_LABELS_DIR, label_file)
    
    if os.path.exists(image_file):
        data_list.append({"image": image_file, "label": label_file})

print(f" Total samples ready: {len(data_list)}")


MONAI TRANSFORMS

In [ ]:
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityRanged, CropForegroundd, RandCropByLabelClassesd,
    RandFlipd, RandRotate90d, RandScaleIntensityd, RandShiftIntensityd,
    EnsureTyped, Compose
)

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(0.5, 0.5, 0.5), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-1000, a_max=3000,
        b_min=0.0, b_max=1.0,
        clip=True
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByLabelClassesd(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        num_classes=7,
        num_samples=4
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=[0, 1, 2]),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
    EnsureTyped(keys=["image", "label"]),
])


DATASET CREATION

In [ ]:
train_ds = CacheDataset(
    data=data_list,
    transform=train_transforms,
    cache_rate=1.0,
    num_workers=0
)

train_loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0       
)


3D UNET MODEL

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=7,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2
).to(device)


LOSS AND OPTIMISER

In [ ]:
import torch
import torch.nn as nn
from monai.losses import DiceLoss
from torch.optim import Adam


class_weights = torch.tensor([0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 1.0], device=device)

ce_loss = nn.CrossEntropyLoss(weight=class_weights)
dice_loss = DiceLoss(to_onehot_y=True, softmax=True)

def combined_loss(outputs, labels):
    return ce_loss(outputs, labels) + dice_loss(outputs, labels)

optimizer = Adam(model.parameters(), lr=1e-4)


In [ ]:
from monai.losses import DiceCELoss

class_weights = torch.tensor([0.05]*6 + [1.0], device=device)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True, weight=class_weights)

from monai.metrics import DiceMetric

dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=True)


TRAINING LOOP

In [ ]:
for epoch in range(start_epoch, MAX_EPOCHS):
    print("=" * 50)
    print(f"Epoch {epoch + 1}/{MAX_EPOCHS}")

    model.train()
    train_loss = 0.0
    train_steps = 0

    for batch_data in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        try:
            inputs = batch_data["image"].to(device)
            labels = batch_data["label"].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_steps += 1
        except RuntimeError as e:
            print(f"Batch skipped due to error: {e}")

    avg_train_loss = train_loss / max(train_steps, 1)
    epoch_loss_values.append(avg_train_loss)
    print(f"Epoch {epoch+1} - Avg Training Loss: {avg_train_loss:.4f}")

    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        val_loss = 0.0
        val_steps = 0
        dice_metric.reset()

        with torch.no_grad():
            for val_data in tqdm(val_loader, desc=f"Validation Epoch {epoch+1}"):
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)

                val_outputs = model(val_inputs)
                loss = loss_function(val_outputs, val_labels)
                val_loss += loss.item()
                val_steps += 1

                val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
                val_labels = [post_label(i) for i in decollate_batch(val_labels)]
                dice_metric(y_pred=val_outputs, y=val_labels)

        avg_val_loss = val_loss / max(val_steps, 1)
        val_loss_values.append(avg_val_loss)

        mean_dice, _ = dice_metric.aggregate()
        mean_dice = mean_dice.item()
        metric_values.append(mean_dice)
        dice_metric.reset()

        print(f"Epoch {epoch+1} - Val Loss: {avg_val_loss:.4f} - Mean Dice: {mean_dice:.4f}")

        if mean_dice > best_metric:
            best_metric = mean_dice
            best_metric_epoch = epoch + 1
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"New Best Model Saved at {BEST_MODEL_PATH}")

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
        "train_loss": epoch_loss_values,
        "val_loss": val_loss_values,
        "val_dice": metric_values,
    }
    torch.save(checkpoint, CHECKPOINT_PATH)
    print(f"Checkpoint saved at {CHECKPOINT_PATH}")

    torch.cuda.empty_cache()

print("=" * 50)
print(f"Training Complete!")
print(f"Best Mean Dice: {best_metric:.4f} at Epoch {best_metric_epoch}")
